In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
from pathlib2 import Path
import graph_construction as gc
import graph_io as gio

c:\Users\alvar\Documents\practice\scripts\graph_data_processing\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
work_dir = Path(r"C:\Users\alvar\Documents\practice\test_repo_spectral_analysis")
path_to_save_spikes = work_dir / "spikes"
path_to_save_spikes.mkdir(exist_ok=True, parents=True)
path_to_save_graph = work_dir / "graph"
path_to_save_graph.mkdir(exist_ok=True, parents=True)

### Инициализация LLM

In [3]:
model_name = "Qwen/Qwen2.5-1.5B" 
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to(device)

### Загрузка датасета

In [4]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", trust_remote_code=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


### Загрузка и токенизация текста

In [5]:
# 2. Параметры эксперимента (Теперь можно ставить огромные числа)
TARGET_TOKEN_LENGTH = 18775 
CHUNK_SIZE = 512            # Размер чанка для экономии памяти GPU
NUM_NEURONS = 562          # Теперь можем собирать данные хоть со всех 4000+ нейронов слоя
selected_neuron_indices = list(range(NUM_NEURONS))

In [6]:
raw_dataset = dataset["train"]
real_long_text = " ".join([item["text"].strip() for item in raw_dataset if len(item["text"].strip()) > 100])
inputs = tokenizer(real_long_text, return_tensors="pt", max_length=TARGET_TOKEN_LENGTH, truncation=True).to(device)

input_ids = inputs["input_ids"][0]
total_tokens = input_ids.shape[0]
print(f"Общая длина текста: {total_tokens} токенов. Размер чанка = {CHUNK_SIZE}.")

Общая длина текста: 18775 токенов. Размер чанка = 512.


In [7]:
# 3. Переменные для сбора данных
dataset_records = []
extracted_data = []
current_token_global_idx = 0

input_ids = inputs["input_ids"][0]
actual_total_tokens = input_ids.shape[0]

firing_matrix = np.zeros((actual_total_tokens, NUM_NEURONS), dtype=np.uint8)
timestamp_vector = np.zeros(actual_total_tokens, dtype=np.float64)
token_index_vector = np.arange(actual_total_tokens, dtype=np.int32)

# Локальный хук, который ловит активации только для текущего маленького чанка
def streaming_hook_perc(module, input, output):
    global current_token_global_idx
    
    # Активации текущего чанка токенов -> [chunk_len, TOTAL_LAYER_NEURONS]
    activations = output.detach() 
    if len(activations.shape) == 3:
        activations = activations[0]
    chunk_len = activations.shape[0]

    # 2. Применяем порог ко ВСЕМУ слою
    thresholds = 4.35
    full_binary_firings = (activations >= thresholds).int()

    # 3. Вырезаем из этой матрицы только интересующие нас нейроны
    subnet_binary_firings = full_binary_firings[:, selected_neuron_indices].cpu().numpy()

    end_idx = current_token_global_idx + chunk_len
    # Проверяем, чтобы индекс не вышел за пределы выделенной матрицы
    if end_idx > firing_matrix.shape[0]:
        # Если из-за особенностей KV-кэша токенов прилетело больше, 
        # берем только столько, сколько влезает в остаток матрицы
        allowed_len = firing_matrix.shape[0] - current_token_global_idx
        if allowed_len <= 0:
            return
        subnet_binary_firings = subnet_binary_firings[:allowed_len]
        end_idx = firing_matrix.shape[0]
        chunk_len = allowed_len

    firing_matrix[current_token_global_idx:end_idx] = subnet_binary_firings
    
    current_token_global_idx = end_idx
    extracted_data.append({
        'chunk_idx': len(extracted_data),
        'activations': activations.clone().cpu().numpy(),
    })


# Регистрация хука
target_layer = model.model.layers[2].mlp.up_proj
hook_handle = target_layer.register_forward_hook(streaming_hook_perc)

In [8]:
# 4. Цикл инференса по чанкам с передачей KV-кэша
past_key_values = None

print("Запуск потокового инференса...")
with torch.no_grad():
    for i in range(0, total_tokens, CHUNK_SIZE):
        # Вырезаем очередной кусок токенов
        chunk_ids = input_ids[i : i + CHUNK_SIZE].unsqueeze(0) # [1, chunk_len]
        
        # Передаем past_key_values, чтобы модель учитывала весь предыдущий контекст
        outputs = model(input_ids=chunk_ids, past_key_values=past_key_values, use_cache=True)
        past_key_values = outputs.past_key_values # Обновляем кэш контекста

hook_handle.remove()

# 6. Проверка результатов
print("\nСбор данных завершен успешно!")
print(f"Форма матрицы firing_matrix: {firing_matrix.shape}")
print(f"Реальная плотность активаций: {firing_matrix.mean() * 100:.5f}%")

np.savez(path_to_save_spikes/"spikes.npz", firing_matrix=firing_matrix, N=562, T=18775)
# np.savez(pat, extracted_data=extracted_data_2)

Запуск потокового инференса...

Сбор данных завершен успешно!
Форма матрицы firing_matrix: (18775, 562)
Реальная плотность активаций: 0.07738%


### Восстановление графа по спайкам

In [9]:
lag_window=40
graph = gc.make_dynamic_network_spikes(firing_matrix, lag_window=lag_window, is_flexible=True)

In [ ]:
undirected_packed_graph, packed_graph, nt = gio.prepare_graph(graph)
graph_path = gio.build_graph_path(path_to_save_graph, "synthetic_dynamic_network", "spikes", is_packed=True)
gio.save_graph(graph_path, undirected_packed_graph, packed_graph, nt)

: 